# Exploring the data downloaded from USDA FoodData Central

See the download here: https://fdc.nal.usda.gov/download-datasets.html

Data available in `.data/`.

Data dictionary available in  `nutrify/data_exploration/data/FoodData_Central_foundation_food_csv_2021-04-28/Download & API Field Descriptions April 2021.pdf`





In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Get Data

In [236]:
# Import databases
food = pd.read_csv("data/FoodData_Central_foundation_food_csv_2026-04-30/food.csv")
food_survey = pd.read_csv("data/FoodData_Central_survey_food_csv_2024-10-31/food.csv")
nutrient = pd.read_csv("data/FoodData_Central_Supporting_Data_csv_2022-10-28/nutrient.csv")
food_nutrient = pd.read_csv("data/FoodData_Central_foundation_food_csv_2026-04-30/food_nutrient.csv")
food_nutrient_survey = pd.read_csv("data/FoodData_Central_survey_food_csv_2024-10-31/food_nutrient.csv")

print(len(food), len(food_survey), len(nutrient), len(food_nutrient), len(food_nutrient_survey))

# Combine food and food_survey and drop columns that don't have a description 
food = pd.concat([food, food_survey], ignore_index=True)
food = food.dropna(subset=["description"])
food["description"] = food["description"].str.lower()
print(f"Combined food rows: {len(food)}")

# Combine food_nutrient and food_nutrient_survey
food_nutrient = pd.concat(
    [food_nutrient, food_nutrient_survey],
    ignore_index=True
)
food_nutrient["nutrient_name"] = food_nutrient["nutrient_id"].map(nutrient.set_index("id")["name"]).str.lower() 
print(f"Combined food nutrient rows: {len(food_nutrient)}")

C:\Users\pyaes\AppData\Local\Temp\ipykernel_7300\2043736291.py:5: DtypeWarning: Columns (0: footnote) have mixed types. Specify dtype option on import or set low_memory=False.
  food_nutrient = pd.read_csv("data/FoodData_Central_foundation_food_csv_2026-04-30/food_nutrient.csv")


87990 5432 474 170469 353015
Combined food rows: 93414
Combined food nutrient rows: 523484


In [3]:
nutrient[nutrient["name"].str.contains("Energy")]

,id,name,unit_name,nutrient_nbr,rank
0,2047,Energy (Atwater General Factors),KCAL,957.0,280.0
1,2048,Energy (Atwater Specific Factors),KCAL,958.0,290.0
9,1008,Energy,KCAL,208.0,300.0
63,1062,Energy,kJ,268.0,400.0


In [4]:
food_nutrient.columns

Index(['id', 'fdc_id', 'nutrient_id', 'amount', 'data_points', 'derivation_id',
       'min', 'max', 'median', 'footnote', 'min_year_acquired',
       'nutrient_name'],
      dtype='str')

In [5]:
len(food_nutrient)

523484

In [6]:
food.head()

,fdc_id,data_type,description,food_category_id,publication_date
0,319874,sample_food,"hummus, sabra classic",16.0,2019-04-01
1,319875,market_acquisition,"hummus, sabra classic",16.0,2019-04-01
2,319876,market_acquisition,"hummus, sabra classic",16.0,2019-04-01
3,319877,sub_sample_food,hummus,16.0,2019-04-01
4,319878,sub_sample_food,hummus,16.0,2019-04-01


In [7]:
food_nutrient.head()

,id,fdc_id,nutrient_id,amount,data_points,derivation_id,min,max,median,footnote,min_year_acquired,nutrient_name
0,2201847,319877,1051,56.30,1.0,1.0,NaN,NaN,NaN,NaN,NaN,water
1,2201845,319877,1002,1.28,1.0,1.0,NaN,NaN,NaN,NaN,NaN,nitrogen
2,2201846,319877,1004,19.00,1.0,1.0,NaN,NaN,NaN,NaN,NaN,total lipid (fat)
3,2201844,319877,1007,1.98,1.0,1.0,NaN,NaN,NaN,NaN,NaN,ash
4,2201852,319878,1091,188.00,1.0,1.0,NaN,NaN,NaN,NaN,NaN,"phosphorus, p"


In [8]:
# How many unique?
unique_descriptions = food["description"].unique()
len(unique_descriptions)

17130

Beautiful, this gives us ~11368 foods to work with as a goal to model. But surely they can be split into less categories?

In [9]:
unique_descriptions[:10]

<ArrowStringArray>
['hummus, sabra classic',                'hummus',         'hummus, other',
    'hummus - nfy12140o',    'hummus - nfy12140p',    'hummus - nfy12140q',
    'hummus - nfy12140r',    'hummus - nfy12140s',    'hummus - nfy12140f',
    'hummus - nfy12140g']
Length: 10, dtype: str

Where do these descriptions come from?

How can we reduce them down to like 10 unique foods and keep it simple...

In [10]:
unique_descriptions[-10:]

<ArrowStringArray>
[                'celery, cooked, as ingredient',
 'dark green vegetables as ingredient in omelet',
              'tomatoes as ingredient in omelet',
      'other vegetables as ingredient in omelet',
               'mirepoix, cooked, as ingredient',
             'vegetables as ingredient in curry',
             'vegetables as ingredient in soups',
             'vegetables as ingredient in stews',
             'sauce as ingredient in hamburgers',
          'industrial oil as ingredient in food']
Length: 10, dtype: str

In [11]:
# Find random indexes of food to explore
import random
random_number = random.randint(0, len(unique_descriptions)-10)
unique_descriptions[random_number:random_number+10]

<ArrowStringArray>
['sauce, pasta, hunts original, canned spaghetti sauce, 26.5 oz container (ca1) - nfy0905jx',
 'sauce, pasta, hunts original, canned spaghetti sauce, 26.5 oz container (ca1) - nfy0905jy',
          'minerals, sauce, pasta, hunts original, canned spaghetti sauce (ca1) - nfy0904z1',
        'proximates, sauce, pasta, hunts original, canned spaghetti sauce (ca1) - nfy0904z3',
                      'sauce, pasta, hunts original, canned spaghetti sauce (nc1) - cy0908j',
 'sauce, pasta, hunts original, canned spaghetti sauce, 26.5 oz container (nc1) - nfy0905kc',
 'sauce, pasta, hunts original, canned spaghetti sauce, 26.5 oz container (nc1) - nfy0905kb',
          'minerals, sauce, pasta, hunts original, canned spaghetti sauce (nc1) - nfy0904zd',
        'proximates, sauce, pasta, hunts original, canned spaghetti sauce (nc1) - nfy0904zf',
                      'sauce, pasta, hunts original, canned spaghetti sauce (ny1) - cy0908f']
Length: 10, dtype: str

### Food Categories

Let's dive into food categories. 

In [12]:
food.head()

,fdc_id,data_type,description,food_category_id,publication_date
0,319874,sample_food,"hummus, sabra classic",16.0,2019-04-01
1,319875,market_acquisition,"hummus, sabra classic",16.0,2019-04-01
2,319876,market_acquisition,"hummus, sabra classic",16.0,2019-04-01
3,319877,sub_sample_food,hummus,16.0,2019-04-01
4,319878,sub_sample_food,hummus,16.0,2019-04-01


In [13]:
unique_categories = food["food_category_id"].unique()
unique_categories

array([1.600e+01, 1.000e+00, 1.300e+01, 1.100e+01, 2.000e+00, 7.000e+00,
       1.200e+01, 6.000e+00, 9.000e+00, 1.800e+01, 4.000e+00, 5.000e+00,
       1.500e+01, 1.900e+01, 2.500e+01, 1.000e+01,       nan, 2.000e+01,
       1.400e+01, 1.700e+01, 9.602e+03, 1.004e+03, 1.002e+03, 1.006e+03,
       1.008e+03, 1.202e+03, 1.902e+03, 1.820e+03, 1.822e+03, 8.412e+03,
       5.802e+03, 9.007e+03, 9.010e+03, 1.206e+03, 1.204e+03, 1.208e+03,
       1.402e+03, 7.220e+03, 9.404e+03, 9.402e+03, 9.999e+03, 8.008e+03,
       8.006e+03, 5.804e+03, 5.502e+03, 1.602e+03, 1.604e+03, 3.602e+03,
       2.502e+03, 3.720e+03, 6.432e+03, 2.002e+03, 2.004e+03, 9.008e+03,
       2.602e+03, 2.604e+03, 2.006e+03, 3.002e+03, 8.002e+03, 2.008e+03,
       2.206e+03, 2.202e+03, 2.204e+03, 2.010e+03, 2.606e+03, 2.608e+03,
       2.402e+03, 2.404e+03, 3.404e+03, 3.004e+03, 8.410e+03, 3.006e+03,
       8.404e+03, 3.202e+03, 3.402e+03, 3.502e+03, 6.411e+03, 3.740e+03,
       3.742e+03, 3.702e+03, 3.704e+03, 3.730e+03, 

19 different food categories... I wonder what these are?

In [14]:
food["food_category_id"].unique()

array([1.600e+01, 1.000e+00, 1.300e+01, 1.100e+01, 2.000e+00, 7.000e+00,
       1.200e+01, 6.000e+00, 9.000e+00, 1.800e+01, 4.000e+00, 5.000e+00,
       1.500e+01, 1.900e+01, 2.500e+01, 1.000e+01,       nan, 2.000e+01,
       1.400e+01, 1.700e+01, 9.602e+03, 1.004e+03, 1.002e+03, 1.006e+03,
       1.008e+03, 1.202e+03, 1.902e+03, 1.820e+03, 1.822e+03, 8.412e+03,
       5.802e+03, 9.007e+03, 9.010e+03, 1.206e+03, 1.204e+03, 1.208e+03,
       1.402e+03, 7.220e+03, 9.404e+03, 9.402e+03, 9.999e+03, 8.008e+03,
       8.006e+03, 5.804e+03, 5.502e+03, 1.602e+03, 1.604e+03, 3.602e+03,
       2.502e+03, 3.720e+03, 6.432e+03, 2.002e+03, 2.004e+03, 9.008e+03,
       2.602e+03, 2.604e+03, 2.006e+03, 3.002e+03, 8.002e+03, 2.008e+03,
       2.206e+03, 2.202e+03, 2.204e+03, 2.010e+03, 2.606e+03, 2.608e+03,
       2.402e+03, 2.404e+03, 3.404e+03, 3.004e+03, 8.410e+03, 3.006e+03,
       8.404e+03, 3.202e+03, 3.402e+03, 3.502e+03, 6.411e+03, 3.740e+03,
       3.742e+03, 3.702e+03, 3.704e+03, 3.730e+03, 

In [15]:
# Get food categories
food_cats = pd.read_csv("data/FoodData_Central_Supporting_Data_csv_2021-04-28/food_category.csv")
food_cats["id"].unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28])

## 10 foods we want

To keep things simple, we will reduce the databases from FoodData Central to 10 different foods.

Why these foods?

Because we have images for those foods ready to go.

```python
# These aren't whole foods so we don't want them yet, let's get another list and get those
ten_foods = ["chicken_curry", 
"chicken_wings", 
"fried_rice", 
"grilled_salmon", 
"humburger", 
"ice_cream", 
"pizza",
"ramen", 
"steak", 
"sushi"]

# We want these... (they're whole foods) 
ten_whole_foods = ["chicken_wings",
    "apple",
    "banana",
    "beef", # steak, etc
    "carrots",
    "egg", # whole egg
    "strawberries",
    "blueberries",
    "mushrooms",
    "honey"
]
```

In [186]:
ten_whole_foods = ['apple',
 'banana',
 'beef', # steak etc
 'blueberries',
 'carrots',
 'chicken_wings',
 'egg', # whole egg
 'honey',
 'mushrooms',
 'strawberries']

# hundred_whole_foods = [ 'apple', 'artichoke', 'bbq_sauce', 'bacon', 'bagel', 
#                 'banana', 'beef', 'beer', 'blueberries', 'bread', 
#                 'broccoli', 'butter', 'cabbage', 'candy', 'cantaloupe', 
#                 'carrots', 'cheese', 'roast_chicken', 'chicken_wings', 
#                 'cocktail', 'coconut', 'coffee', 'cookie', 'corn_chips', 
#                 'cream', 'cucumber', 'doughnut', 'egg', 'fish', 'fries', 
#                 'grape', 'guacamole', 'hamburger', 'honey', 'ice_cream', 
#                 'lemon', 'lime', 'lobster', 'mango', 'milk', 'muffin', 
#                 'mushrooms', 'olive_oil', 'olives', 'onion', 'orange', 
#                 'orange_juice', 'pancake', 'pasta', 'pastry', 'pear', 
#                 'bell_pepper', 'pineapple', 'pizza', 'pomegranate', 'popcorn', 
#                 'potato', 'prawns', 'pretzel', 'pumpkin', 'radish', 'rice', 
#                 'salad', 'salt', 'sandwich', 'sausages', 'soft_drink', 
#                 'spinach', 'squid', 'strawberries', 'sushi', 'tea', 
#                 'tomato', 'tomato_sauce', 'waffle', 'watermelon', 'wine', 
#                 'zucchini', 'avocado', 'asparagus', 'beetroot', 'cauliflower', 
#                 'celery', 'cherries', 'corn', 'dates', 'dragon_fruit', 
#                 'durian', 'eggplant', 'garlic', 'ginger', 'green_beans', 
#                 'kiwi', 'lettuce', 'papaya', 'peach', 'peas', 
#                 'raspberries', 'sweet_potato', 'turnip' ]

hundred_whole_foods = [
    # =========================
    # FRUITS - 25
    # =========================
    'apple',              # original
    'banana',             # original
    'blueberries',        # original
    'strawberries',       # original
    'orange',
    'mango',
    'pineapple',
    'watermelon',
    'grapes',
    'pear',
    'peach',
    'kiwi',
    'papaya',
    'avocado',
    'pomegranate',
    'cherries',
    'raspberries',
    'blackberries',
    'cantaloupe',
    'grapefruit',
    'plum',
    'apricot',
    'figs',
    'dates',
    'coconut',

    # =========================
    # VEGETABLES - 25
    # =========================
    'carrots',            # original
    'mushrooms',          # original
    'broccoli',
    'cauliflower',
    'cabbage',
    'spinach',
    'lettuce',
    'cucumber',
    'tomato',
    'bell_pepper',
    'onion',
    'garlic',
    'potato',
    'sweet_potato',
    'pumpkin',
    'zucchini',
    'eggplant',
    'asparagus',
    'celery',
    'green_beans',
    'peas',
    'corn',
    'beetroot',
    'radish',
    'turnip',

    # =========================
    # MEAT / SEAFOOD / EGG - 20
    # =========================
    'beef',               # original
    'chicken_wings',      # original
    'egg',                # original
    'chicken_breast',
    'chicken_thigh',
    'pork_chop',
    'bacon',
    'turkey_breast',
    'salmon',
    'tuna',
    'shrimp',
    'crab',
    'lobster',
    'squid',
    'cod',
    'sardines',
    'tilapia',
    'lamb_chop',
    'duck_breast',
    'ham',

    # =========================
    # LEGUMES / NUTS / SEEDS - 15
    # =========================
    'tofu',
    'lentils',
    'chickpeas',
    'kidney_beans',
    'black_beans',
    'peanuts',
    'almonds',
    'cashews',
    'walnuts',
    'pistachios',
    'sunflower_seeds',
    'pumpkin_seeds',
    'sesame_seeds',
    'chia_seeds',
    'soybeans',

    # =========================
    # DAIRY / GRAINS / OTHER - 15
    # =========================
    'honey',              # original
    'milk',
    'yogurt',
    'cheddar_cheese',
    'mozzarella_cheese',
    'butter',
    'rice',
    'brown_rice',
    'oats',
    'bread',
    'pasta',
    'noodles',
    'bagel',
    'popcorn',
    'peanut_butter'
]

In [187]:
food.head()

AttributeError: 'str' object has no attribute 'head'

In [188]:
# Foundation food is the ground truth for a certain type of food, excludes some details about the food
# E.g. the data_type foundation_food for Chicken will the the original unique ID for chicken
foundation_food = food[(food["data_type"] == "foundation_food") | (food["data_type"] == "survey_fndds_food")]
len(foundation_food)

TypeError: string indices must be integers, not 'str'

In [189]:
foundation_food[foundation_food["description"].str.contains("blue")]

,fdc_id,data_type,description,food_category_id,publication_date
42186,2263889,foundation_food,"blueberries, raw",9.0,2022-04-28
43441,2346411,foundation_food,"blueberries, raw",9.0,2022-10-28
60660,2684446,foundation_food,"crustaceans, crab, blue swimming, lump, pasteu...",15.0,2024-04-18
88312,2705705,survey_fndds_food,"cheese, blue or roquefort",1602.0,2022-10-28
90605,2707998,survey_fndds_food,"pie, blueberry",5502.0,2022-10-28
91805,2709198,survey_fndds_food,"blueberries, dried",6016.0,2022-10-28
91882,2709275,survey_fndds_food,"blueberries, raw",6011.0,2022-10-28
91884,2709277,survey_fndds_food,"blueberries, frozen",6011.0,2022-10-28
91885,2709278,survey_fndds_food,blueberry pie filling,8806.0,2022-10-28
91930,2709323,survey_fndds_food,blueberry juice,7006.0,2022-10-28


In [190]:
foundation_foods = foundation_food["description"]
foundation_foods[20:40]

4153               peanut butter, smooth style, with salt
4329                             cheese, parmesan, grated
4491    cheese, pasteurized process, american, vitamin...
4580    grapefruit juice, white, canned or bottled, un...
4723                                 peaches, yellow, raw
4817    seeds, sunflower seed kernels, dry roasted, wi...
4951      sausage, italian, pork, mild, cooked, pan-fried
5164                  bread, white, commercially prepared
5285          sausage, turkey, breakfast links, mild, raw
5428                                        cheese, swiss
5489    kale, frozen, cooked, boiled, drained, without...
5751    carrots, frozen, unprepared (includes foods fo...
5991                            mustard, prepared, yellow
6198                                figs, dried, uncooked
6339                                kiwifruit, green, raw
6491                              melons, cantaloupe, raw
6650                                      nectarines, raw
6794    orange

In [191]:
# Found a list of the foundation foods we're going to start with!
foundation_foods_list = list(foundation_foods)
for food in foundation_foods_list:
    if "blue" in food:
        print(food)

blueberries, raw
blueberries, raw
crustaceans, crab, blue swimming, lump, pasteurized, refrigerated
cheese, blue or roquefort
pie, blueberry
blueberries, dried
blueberries, raw
blueberries, frozen
blueberry pie filling
blueberry juice
blue or roquefort cheese dressing
blue or roquefort cheese dressing, light
blue or roquefort cheese dressing, fat free
blueberry syrup


In [192]:
# food.loc[(food["description"].str.contains("chicken", case=False)) & (food["description"].str.contains("drumstick", case=False))][-10:]
# Find chicken in foundation food
for food in foundation_foods:
    if "chicken" in food.lower():
        print(food)

chicken, broilers or fryers, drumstick, meat only, cooked, braised
chicken, broiler or fryers, breast, skinless, boneless, meat only, cooked, braised
chicken, ground, with additives, raw
chicken, breast, boneless, skinless, raw
chicken, thigh, boneless, skinless, raw
chicken, drumstick, meat and skin, raw
chicken, thigh, meat and skin, raw
chicken, wing, meat and skin, raw
chicken, breast, meat and skin, raw
lunchmeat, chicken breast, sliced
mock chicken legs
chicken, ns as to part and cooking method, ns as to skin eaten
chicken, ns as to part and cooking method, skin eaten
chicken, ns as to part and cooking method, skin not eaten
chicken, ns as to part, baked, broiled, or roasted, ns as to skin eaten
chicken, ns as to part, baked, broiled, or roasted, skin eaten
chicken, ns as to part, baked, broiled, or roasted, skin not eaten
chicken, ns as to part, rotisserie, ns as to skin eaten
chicken, ns as to part, rotisserie, skin eaten
chicken, ns as to part, rotisserie, skin not eaten
chick

In [193]:
chicken_wing_id = int(foundation_food.loc[foundation_food["description"].str.contains("Chicken", case=False)].iloc[0]["fdc_id"])
chicken_wing_id

331897

In [194]:
apple_id = int(foundation_food.loc[foundation_food["description"].str.contains("Apple", case=False)].iloc[0]["fdc_id"])
apple_id

1105430

In [195]:
food_nutrient[food_nutrient["fdc_id"] == chicken_wing_id]

,id,fdc_id,nutrient_id,amount,data_points,derivation_id,min,max,median,footnote,min_year_acquired,nutrient_name
41714,2259068,331897,1303,0.003,5.0,1.0,0.002,0.004,0.003,NaN,2010.0,tfa 16:1 t
41715,2259065,331897,1280,0.008,5.0,1.0,0.008,0.009,0.008,NaN,2010.0,pufa 22:5 n-3 (dpa)
41716,2259076,331897,1404,0.045,5.0,1.0,0.035,0.059,0.042,NaN,2010.0,"pufa 18:3 n-3 c,c,c (ala)"
41717,2259059,331897,1261,0.002,5.0,1.0,0.001,0.003,0.002,NaN,2010.0,sfa 8:0
41718,2259106,331897,1109,0.170,1.0,1.0,NaN,NaN,0.170,NaN,2010.0,vitamin e (alpha-tocopherol)
...,...,...,...,...,...,...,...,...,...,...,...,...
41806,2259112,331897,1167,5.050,5.0,1.0,4.890,5.240,5.050,NaN,2010.0,niacin
41807,2259074,331897,1329,0.021,NaN,4.0,NaN,NaN,NaN,NaN,NaN,"fatty acids, total trans-monoenoic"
41808,2259138,331897,1330,0.008,NaN,4.0,NaN,NaN,NaN,NaN,NaN,"fatty acids, total trans-dienoic"
41809,13338545,331897,2047,149.000,NaN,1.0,NaN,NaN,NaN,NaN,NaN,energy (atwater general factors)


## Get protein, carb, fat IDs

See this document for info on foundation foods and their nutrients - https://fdc.nal.usda.gov/docs/Foundation_Foods_Documentation_Apr2021.pdf

* Carbohydrate, by difference = total carbohydrates


In [196]:
nutrient[(nutrient["name"].str.contains("protein", case=False)) | \
         (nutrient["name"].str.contains("carbohydrate", case=False)) | \
         (nutrient["name"].str.contains("fat", case=False)) | \
         (nutrient["name"].str.contains("energy", case=False))]

,id,name,unit_name,nutrient_nbr,rank
0,2047,Energy (Atwater General Factors),KCAL,957.0,280.0
1,2048,Energy (Atwater Specific Factors),KCAL,958.0,290.0
4,1003,Protein,G,203.0,600.0
5,1004,Total lipid (fat),G,204.0,800.0
6,1005,"Carbohydrate, by difference",G,205.0,1110.0
9,1008,Energy,KCAL,208.0,300.0
50,1049,"Solids, non-fat",G,253.0,999999.0
51,1050,"Carbohydrate, by summation",G,205.2,1120.0
54,1053,Adjusted Protein,G,257.0,700.0
63,1062,Energy,kJ,268.0,400.0


In [197]:
target_nutrients = nutrient[nutrient["name"].isin(["Protein", "Total lipid (fat)", "Carbohydrate, by difference", "Energy (Atwater Specific Factors)"])]

# target_nutrients = target_nutrients[
#     ~((nutrient["name"] == "Energy") & (nutrient["unit_name"] == "kJ"))
# ]
target_nutrients

,id,name,unit_name,nutrient_nbr,rank
1,2048,Energy (Atwater Specific Factors),KCAL,958.0,290.0
4,1003,Protein,G,203.0,600.0
5,1004,Total lipid (fat),G,204.0,800.0
6,1005,"Carbohydrate, by difference",G,205.0,1110.0


In [198]:
target_nutrient_dict = {
    # Modern IDs (foundation foods)
    1003: "protein",
    1004: "fat",
    1005: "carbohydrate",
    1008: "energy_kcal",      # standard Energy (kcal)
    2048: "energy_kcal",      # Energy (Atwater Specific Factors)
    # Legacy IDs (survey/FNDDS foods)
    203: "protein",
    204: "fat",
    205: "carbohydrate",
    208: "energy_kcal",
}
target_nutrient_dict

{1003: 'protein',
 1004: 'fat',
 1005: 'carbohydrate',
 1008: 'energy_kcal',
 2048: 'energy_kcal',
 203: 'protein',
 204: 'fat',
 205: 'carbohydrate',
 208: 'energy_kcal'}

## Get target food protein, fat, carbohydrates

We want to now index on the target foods and the target nutrients and retrieve their values for each food/nutrient.

E.g.

```python
{"food_1": {"protein": 100,
            "carbohydrate": 50,
            "fat": 20},
 "food_2": ...

...}
```

In [199]:
list(target_nutrient_dict.keys())

[1003, 1004, 1005, 1008, 2048, 203, 204, 205, 208]

In [200]:
food_nutrient

,id,fdc_id,nutrient_id,amount,data_points,derivation_id,min,max,median,footnote,min_year_acquired,nutrient_name
0,2201847,319877,1051,56.30,1.0,1.0,NaN,NaN,NaN,NaN,NaN,water
1,2201845,319877,1002,1.28,1.0,1.0,NaN,NaN,NaN,NaN,NaN,nitrogen
2,2201846,319877,1004,19.00,1.0,1.0,NaN,NaN,NaN,NaN,NaN,total lipid (fat)
3,2201844,319877,1007,1.98,1.0,1.0,NaN,NaN,NaN,NaN,NaN,ash
4,2201852,319878,1091,188.00,1.0,1.0,NaN,NaN,NaN,NaN,NaN,"phosphorus, p"
...,...,...,...,...,...,...,...,...,...,...,...,...
523479,34489112,2710814,208,892.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
523480,34489151,2710814,601,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
523481,34489135,2710814,337,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
523482,34489121,2710814,304,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [201]:
food_nutrient[(food_nutrient["nutrient_id"].isin(list(target_nutrient_dict.keys())))]

,id,fdc_id,nutrient_id,amount,data_points,derivation_id,min,max,median,footnote,min_year_acquired,nutrient_name
2,2201846,319877,1004,19.00,1.0,1.0,NaN,NaN,NaN,NaN,NaN,total lipid (fat)
16,2201859,319882,1004,18.70,1.0,1.0,NaN,NaN,NaN,NaN,NaN,total lipid (fat)
28,2201873,319892,1004,16.60,1.0,1.0,NaN,NaN,NaN,NaN,NaN,total lipid (fat)
43,2201886,319899,1004,19.10,1.0,1.0,NaN,NaN,NaN,NaN,NaN,total lipid (fat)
97,2201942,319908,1004,18.20,1.0,1.0,NaN,NaN,NaN,NaN,NaN,total lipid (fat)
...,...,...,...,...,...,...,...,...,...,...,...,...
523400,34489046,2710813,205,17.14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
523452,34489111,2710814,205,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
523464,34489110,2710814,204,100.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
523469,34489109,2710814,203,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [202]:
unique_nutrients = food_nutrient[
    food_nutrient["nutrient_id"].isin(list(target_nutrient_dict.keys()))
]["nutrient_id"].unique()

print(unique_nutrients)

[1004 1003 1008 1005 2048  204  208  205  203]


In [203]:
food_nutrient.dtypes

id                     int64
fdc_id                 int64
nutrient_id            int64
amount               float64
data_points          float64
derivation_id        float64
min                  float64
max                  float64
median               float64
footnote              object
min_year_acquired    float64
nutrient_name            str
dtype: object

In [204]:
# Find nutrition for chicken_wing_id (protein, fat, carb)
food_nutrient[(food_nutrient["fdc_id"] == chicken_wing_id) & (food_nutrient["nutrient_id"].isin(list(target_nutrient_dict.keys())))]

,id,fdc_id,nutrient_id,amount,data_points,derivation_id,min,max,median,footnote,min_year_acquired,nutrient_name
41746,2259100,331897,1008,156.00,NaN,49.0,NaN,NaN,NaN,NaN,NaN,energy
41750,2259098,331897,1004,5.95,6.0,1.0,5.54,6.33,5.93,NaN,2010.0,total lipid (fat)
41782,2259079,331897,1003,23.90,NaN,49.0,23.00,24.60,24.10,NaN,NaN,protein
41793,2259099,331897,1005,0.00,NaN,49.0,NaN,NaN,NaN,NaN,NaN,"carbohydrate, by difference"
41810,13338447,331897,2048,156.00,NaN,49.0,NaN,NaN,NaN,NaN,NaN,energy (atwater specific factors)


In [205]:
sorted(list(foundation_foods))

['abalone',
 'adobo, with noodles',
 'adobo, with rice',
 'agave liquid sweetener',
 'alaska pollock, raw',
 'alcoholic coffee drink',
 'alcoholic malt beverage',
 'alcoholic malt beverage, sweetened',
 'alfalfa sprouts, raw',
 'alfredo sauce',
 'alfredo sauce with added vegetables',
 'alfredo sauce with meat',
 'alfredo sauce with meat and added vegetables',
 'alfredo sauce with poultry',
 'alfredo sauce with poultry and added vegetables',
 'alfredo sauce with seafood',
 'alfredo sauce with seafood and added vegetables',
 'almond butter',
 'almond butter and jelly sandwich, on wheat bread',
 'almond butter and jelly sandwich, on white bread',
 'almond butter sandwich, on wheat bread',
 'almond butter sandwich, on white bread',
 'almond butter, creamy',
 'almond butter, lower sodium',
 'almond chicken',
 'almond milk, chocolate',
 'almond milk, nfs',
 'almond milk, sweetened',
 'almond milk, unsweetened',
 'almond milk, unsweetened, plain, refrigerated',
 'almond milk, unsweetened, pla

In [206]:
ten_whole_foods = ["chicken_wings",
    "apple",
    "banana",
    "beef", # steak, etc
    "carrots",
    "egg", # whole egg
    "strawberries",
    "blueberries",
    "mushrooms",
    "honey"
]

In [207]:
hundred_whole_foods

['apple',
 'banana',
 'blueberries',
 'strawberries',
 'orange',
 'mango',
 'pineapple',
 'watermelon',
 'grapes',
 'pear',
 'peach',
 'kiwi',
 'papaya',
 'avocado',
 'pomegranate',
 'cherries',
 'raspberries',
 'blackberries',
 'cantaloupe',
 'grapefruit',
 'plum',
 'apricot',
 'figs',
 'dates',
 'coconut',
 'carrots',
 'mushrooms',
 'broccoli',
 'cauliflower',
 'cabbage',
 'spinach',
 'lettuce',
 'cucumber',
 'tomato',
 'bell_pepper',
 'onion',
 'garlic',
 'potato',
 'sweet_potato',
 'pumpkin',
 'zucchini',
 'eggplant',
 'asparagus',
 'celery',
 'green_beans',
 'peas',
 'corn',
 'beetroot',
 'radish',
 'turnip',
 'beef',
 'chicken_wings',
 'egg',
 'chicken_breast',
 'chicken_thigh',
 'pork_chop',
 'bacon',
 'turkey_breast',
 'salmon',
 'tuna',
 'shrimp',
 'crab',
 'lobster',
 'squid',
 'cod',
 'sardines',
 'tilapia',
 'lamb_chop',
 'duck_breast',
 'ham',
 'tofu',
 'lentils',
 'chickpeas',
 'kidney_beans',
 'black_beans',
 'peanuts',
 'almonds',
 'cashews',
 'walnuts',
 'pistachios',


## Get ten whole foods `food_id`

Everything except blueberries and honey are available in `foundation_food`. 

For blueberries and honey, we'll have to dig into the survery data: `data_exploration/data/FoodData_Central_survey_food_csv_2020-10-30`

In [208]:
# Get all food ids from foundation_food (honey and blueberries in another dataset)
target_whole_foods = ['apple', # removed chicken wings... can come back later...
 'banana',
 'beef',
 'blueberries',
 'carrots',
 'chicken',
 'egg',
 'honey',
 'strawberries',
 'mushrooms']

target_whole_foods = [

    # =========================
    # FRUITS - 25
    # =========================
    'apple',              # original
    'banana',             # original
    'blueberries',        # original
    'strawberries',       # original
    'orange',
    'mango',
    'pineapple',
    'watermelon',
    'grapes',
    'pear',
    'peach',
    'kiwi',
    'papaya',
    'avocado',
    'pomegranate',
    'cherries',
    'raspberries',
    'blackberries',
    'cantaloupe',
    'grapefruit',
    'plum',
    'apricot',
    'figs',
    'dates',
    'coconut',

    # =========================
    # VEGETABLES - 25
    # =========================
    'carrots',            # original
    'mushrooms',          # original
    'broccoli',
    'cauliflower',
    'cabbage',
    'spinach',
    'lettuce',
    'cucumber',
    'tomato',
    'bell_pepper',
    'onion',
    'garlic',
    'potato',
    'sweet_potato',
    'pumpkin',
    'zucchini',
    'eggplant',
    'asparagus',
    'celery',
    'green_beans',
    'peas',
    'corn',
    'beetroot',
    'radish',
    'turnip',

    # =========================
    # MEAT / SEAFOOD / EGG - 20
    # =========================
    'beef',               # original
    'chicken_wings',      # original
    'egg',                # original
    'chicken_breast',
    'chicken_thigh',
    'pork_chop',
    'bacon',
    'turkey_breast',
    'salmon',
    'tuna',
    'shrimp',
    'crab',
    'lobster',
    'squid',
    'cod',
    'sardines',
    'tilapia',
    'lamb_chop',
    'duck_breast',
    'ham',

    # =========================
    # LEGUMES / NUTS / SEEDS - 15
    # =========================
    'tofu',
    'lentils',
    'chickpeas',
    'kidney_beans',
    'black_beans',
    'peanuts',
    'almonds',
    'cashews',
    'walnuts',
    'pistachios',
    'sunflower_seeds',
    'pumpkin_seeds',
    'sesame_seeds',
    'chia_seeds',
    'soybeans',

    # =========================
    # DAIRY / GRAINS / OTHER - 15
    # =========================
    'honey',              # original
    'milk',
    'yogurt',
    'cheddar_cheese',
    'mozzarella_cheese',
    'butter',
    'rice',
    'brown_rice',
    'oats',
    'bread',
    'pasta',
    'noodles',
    'bagel',
    'popcorn',
    'peanut_butter'
]

In [209]:
len(target_whole_foods)

100

In [229]:
# str.contains can search on regex - https://stackoverflow.com/a/17973255/7900723
pattern = "|".join([f"(?i){food}" for food in target_whole_foods])
pattern

'(?i)apple|(?i)banana|(?i)blueberries|(?i)strawberries|(?i)orange|(?i)mango|(?i)pineapple|(?i)watermelon|(?i)grapes|(?i)pear|(?i)peach|(?i)kiwi|(?i)papaya|(?i)avocado|(?i)pomegranate|(?i)cherries|(?i)raspberries|(?i)blackberries|(?i)cantaloupe|(?i)grapefruit|(?i)plum|(?i)apricot|(?i)figs|(?i)dates|(?i)coconut|(?i)carrots|(?i)mushrooms|(?i)broccoli|(?i)cauliflower|(?i)cabbage|(?i)spinach|(?i)lettuce|(?i)cucumber|(?i)tomato|(?i)bell_pepper|(?i)onion|(?i)garlic|(?i)potato|(?i)sweet_potato|(?i)pumpkin|(?i)zucchini|(?i)eggplant|(?i)asparagus|(?i)celery|(?i)green_beans|(?i)peas|(?i)corn|(?i)beetroot|(?i)radish|(?i)turnip|(?i)beef|(?i)chicken_wings|(?i)egg|(?i)chicken_breast|(?i)chicken_thigh|(?i)pork_chop|(?i)bacon|(?i)turkey_breast|(?i)salmon|(?i)tuna|(?i)shrimp|(?i)crab|(?i)lobster|(?i)squid|(?i)cod|(?i)sardines|(?i)tilapia|(?i)lamb_chop|(?i)duck_breast|(?i)ham|(?i)tofu|(?i)lentils|(?i)chickpeas|(?i)kidney_beans|(?i)black_beans|(?i)peanuts|(?i)almonds|(?i)cashews|(?i)walnuts|(?i)pistachios

In [ ]:
foundation_food[foundation_food["description"].str.contains(pattern, case=False)].sort_values(by=["description"])

In [234]:
foundation_food[foundation_food["description"].str.contains("honey")]

,fdc_id,data_type,description,food_category_id,publication_date
20191,1105547,foundation_food,"apples, honeycrisp, with skin, raw",9.0,2020-10-30
20547,1750343,foundation_food,"apples, honeycrisp, with skin, raw",9.0,2020-10-30
63976,2710816,foundation_food,"melons, honeydew, raw",9.0,2024-10-31
90098,2707491,survey_fndds_food,"almonds, honey roasted",2804.0,2022-10-28
90105,2707498,survey_fndds_food,"cashews, honey roasted",2804.0,2022-10-28
90118,2707511,survey_fndds_food,"mixed nuts, honey roasted",2804.0,2022-10-28
90127,2707520,survey_fndds_food,"peanuts, honey roasted",2804.0,2022-10-28
90132,2707525,survey_fndds_food,"pecans, honey roasted",2804.0,2022-10-28
90138,2707531,survey_fndds_food,"walnuts, excluding honey roasted",2804.0,2022-10-28
90139,2707532,survey_fndds_food,"walnuts, honey roasted",2804.0,2022-10-28


In [216]:
# Found this earlier
chicken_wing_id

331897

In [237]:
# Build a mapping: for each food, find the best fdc_id match
import pandas as pd

food_id_map = {}

for food_name in target_whole_foods:
    matches = food[food["description"].str.contains(
        food_name.replace("_", " "), case=False, na=False
    )][["fdc_id", "description", "data_type"]].copy()
    
    if len(matches) == 0:
        print(f"❌ {food_name:20s} → NO MATCH FOUND")
        continue
    
    priority = {"foundation_food": 1, "survey_fndds_food": 2}
    matches["priority"] = matches["data_type"].map(priority).fillna(3)
    matches = matches.sort_values("priority")
    
    top5 = matches.head(5)
    print(f"\n{'='*60}")
    print(f"🍽 {food_name} — top {len(top5)} matches:")
    for _, row in top5.iterrows():
        print(f"   ID: {row['fdc_id']:>8}  |  {row['description']:50s}  ({row['data_type']})")
    
    best = matches.iloc[0]
    food_id_map[food_name] = int(best["fdc_id"])

print(f"\n{'='*60}")
print(f"\nSummary: found IDs for {len(food_id_map)} / {len(target_whole_foods)} foods")


🍽 apple — top 5 matches:
   ID:  1750342  |  apples, granny smith, with skin, raw                (foundation_food)
   ID:  1750343  |  apples, honeycrisp, with skin, raw                  (foundation_food)
   ID:  1105430  |  apples, red delicious, with skin, raw               (foundation_food)
   ID:  1105547  |  apples, honeycrisp, with skin, raw                  (foundation_food)
   ID:  1750339  |  apples, red delicious, with skin, raw               (foundation_food)

🍽 banana — top 5 matches:
   ID:  2747660  |  peppers, banana or hungarian wax, seeded, raw       (foundation_food)
   ID:  1105073  |  bananas, overripe, raw                              (foundation_food)
   ID:  1105314  |  bananas, ripe and slightly ripe, raw                (foundation_food)
   ID:   790991  |  bananas, ripe and slightly ripe, raw                (foundation_food)
   ID:   790774  |  bananas, overripe, raw                              (foundation_food)

🍽 blueberries — top 5 matches:
   ID:  2263889

In [241]:
# Broader searches for the 4 missing
broad = {
    "turkey_breast": "turkey",
    "lamb_chop": "lamb",
    "duck_breast": "duck",
    "dates": "date",
}
for name, term in broad.items():
    matches = food[food["description"].str.contains(rf"\b{term}\b", case=False, na=False, regex=True)][
        ["fdc_id", "description", "data_type"]
    ].head(3)
    print(f"\n--- {name} (searched: '{term}') ---")
    if len(matches) == 0:
        print("  ❌ NO MATCH")
    for _, row in matches.iterrows():
        print(f"  ID: {row['fdc_id']:>8} | {row['description']:55s} | {row['data_type']}")


--- turkey_breast (searched: 'turkey') ---
  ID:   325872 | turkey breakfast sausage, honeysuckle white             | sample_food
  ID:   325873 | turkey breakfast sausage, honeysuckle white             | market_acquisition
  ID:   325874 | turkey breakfast sausage, honeysuckle white             | market_acquisition

--- lamb_chop (searched: 'lamb') ---
  ID:  2727570 | lamb, ground, raw                                       | foundation_food
  ID:  2727651 | lamb, ground, raw                                       | sample_food
  ID:  2727653 | lamb, ground, raw                                       | sample_food

--- duck_breast (searched: 'duck') ---
  ID:  2706137 | duck, cooked, skin eaten                                | survey_fndds_food
  ID:  2706138 | duck, cooked, skin not eaten                            | survey_fndds_food
  ID:  2706139 | duck, roasted, skin eaten                               | survey_fndds_food

--- dates (searched: 'date') ---
  ID:  2708115 | breakfas

In [242]:
# Map foods to food_id (these have been filtered from larger quantities to smaller quantities)
# For example, if there were 5 kinds of apple, only one was chosen
whole_foods_id_map = {1750339: "apple", # red delicious
    1105314: "banana", # Bananas, ripe and slightly ripe, raw
    1102702: "blueberries", # blueberries, raw	
    746763: "beef", # t-bone steak 
    746764: "carrots", # frozen unprepared
    331897: "chicken_wings", # Chicken, broilers or fryers, drumstick, meat o...	
    329490: "egg", # Egg, whole, dried	
    1103956: "honey", # Honey
    1750347: "mushrooms", # Mushrooms, white button
    747448: "strawberries" # strawberries, raw
}
whole_foods_id_map = {
    # =========================
    # FRUITS - 25
    # =========================
    1750339: "apple",           # apples, red delicious, with skin, raw
    1105314: "banana",          # bananas, ripe and slightly ripe, raw
    2263889: "blueberries",     # blueberries, raw
    2346409: "strawberries",    # strawberries, raw
    746771: "orange",           # oranges, raw, navels
    2710833: "mango",           # mango, tommy atkins, peeled, raw
    2346398: "pineapple",       # pineapple, raw
    2747675: "watermelon",      # watermelon, seedless, flesh only, raw
    2263890: "grapes",          # grapes, red, seedless, raw
    746773: "pear",             # pears, raw, bartlett
    325430: "peach",            # peaches, yellow, raw
    327046: "kiwi",             # kiwifruit, green, raw
    2709246: "papaya",          # papaya, raw
    2710824: "avocado",         # avocado, hass, peeled, raw
    2709267: "pomegranate",     # pomegranate, raw
    2346399: "cherries",        # cherries, sweet, dark red, raw
    2263888: "raspberries",     # raspberries, raw
    2727581: "blackberries",    # blackberries, raw
    327198: "cantaloupe",       # melons, cantaloupe, raw
    2758977: "grapefruit",      # grapefruit, raw
    2710837: "plum",            # plum, black, with skin, raw
    2710815: "apricot",         # apricot, with skin, raw
    746768: "figs",             # figs, dried, uncooked
    2709203: "dates",           # date
    2707501: "coconut",         # coconut, packaged

    # =========================
    # VEGETABLES - 25
    # =========================
    746764: "carrots",          # carrots, frozen, unprepared
    1750347: "mushrooms",       # mushrooms, white button
    747447: "broccoli",         # broccoli, raw
    2685573: "cauliflower",     # cauliflower, raw
    2346407: "cabbage",         # cabbage, green, raw
    1750352: "spinach",         # spinach, baby
    2346388: "lettuce",         # lettuce, iceberg, raw
    2346406: "cucumber",        # cucumber, with peel, raw
    2685578: "tomato",          # tomatoes, whole, canned
    2258588: "bell_pepper",     # peppers, bell, green, raw
    1104962: "onion",           # onions, white, raw
    1104647: "garlic",          # garlic, raw
    2346401: "potato",          # potatoes, russet, without skin, raw
    2346404: "sweet_potato",    # sweet potatoes, orange flesh, without skin, raw
    2727578: "pumpkin",         # squash, pie pumpkin, peeled, seeded, raw
    2685568: "zucchini",        # squash, summer, green, zucchini, raw
    2685577: "eggplant",        # eggplant, raw
    2710823: "asparagus",       # asparagus, green, raw
    2346405: "celery",          # celery, raw
    2709853: "green_beans",     # green beans, frozen, cooked, no added fat
    2644291: "peas",            # peas, green, sweet, canned
    2710826: "corn",            # corn, sweet, yellow and white kernels, fresh, raw
    2685576: "beetroot",        # beets, raw
    2747665: "radish",          # radishes, red, raw
    2747674: "turnip",          # turnips, raw

    # =========================
    # MEAT / SEAFOOD / EGG - 20
    # =========================
    746763: "beef",             # beef, t-bone steak
    331897: "chicken_wings",    # chicken, broilers or fryers, drumstick, meat only
    329490: "egg",              # egg, whole, dried
    2759004: "chicken_breast",  # lunchmeat, chicken breast, sliced
    2706027: "chicken_thigh",   # chicken thigh, ns as to cooking method, skin eaten
    2646168: "pork_chop",       # pork, loin, boneless, raw
    749420: "bacon",            # pork, cured, bacon, cooked, restaurant
    2514747: "turkey_breast",   # turkey, ground, 93% lean/7% fat, raw
    2684440: "salmon",          # fish, salmon, sockeye, wild caught, raw
    2747673: "tuna",            # tuna, ahi or yellowfin, frozen, wild caught
    2684443: "shrimp",          # crustaceans, shrimp, farm raised, raw
    2684446: "crab",            # crustaceans, crab, blue swimming, lump, pasteurized
    2747657: "lobster",         # lobster, tail only, frozen, wild caught
    2747671: "squid",           # squid (calamari), frozen, tubes only
    2684444: "cod",             # fish, cod, atlantic, wild caught, raw
    2706293: "sardines",        # fish, sardines, canned
    2684442: "tilapia",         # fish, tilapia, farm raised, raw
    2727570: "lamb_chop",       # lamb, ground, raw
    2706137: "duck_breast",     # duck, cooked, skin eaten
    2759002: "ham",             # lunchmeat, ham, black forest, sliced

    # =========================
    # LEGUMES / NUTS / SEEDS - 15
    # =========================
    # tofu → not in dataset; use a substitute
    2644283: "Lentils, dry",            # PLACEHOLDER: currently lentils. Search for "soybean curd" or replace
    2644283: "lentils",         # lentils, dry
    2644282: "chickpeas",       # chickpeas, (garbanzo beans, bengal gram), dry
    2707379: "kidney_beans",    # kidney beans, nfs
    2707359: "black_beans",     # black beans, nfs
    2515376: "peanuts",         # peanuts, raw
    2346393: "almonds",         # nuts, almonds, whole, raw
    2707497: "cashews",         # cashews, unsalted
    2346394: "walnuts",         # nuts, walnuts, english, halves, raw
    2515379: "pistachios",      # nuts, pistachio nuts, raw
    2707585: "sunflower_seeds", # sunflower seeds, nfs
    2515380: "pumpkin_seeds",   # seeds, pumpkin seeds (pepitas), raw
    2707586: "sesame_seeds",    # sesame seeds
    2710819: "chia_seeds",      # chia seeds, dry, raw
    2707388: "soybeans",        # soybeans, cooked

    # =========================
    # DAIRY / GRAINS / OTHER - 15
    # =========================
    2710281: "honey",           # honey
    746782: "milk",             # milk, whole, 3.25% milkfat
    2259793: "yogurt",          # yogurt, plain, whole milk
    325198: "cheddar_cheese",   # cheese, pasteurized process, american
    # mozzarella → not in dataset; use provolone as closest
    2647440: "Cheese, provolone, sliced", # cheese, provolone, sliced
    790508: "butter",           # butter, stick, salted
    2512381: "rice",            # rice, white, long grain, unenriched, raw
    2709033: "brown_rice",      # beans and brown rice
    2346396: "oats",            # oats, whole grain, rolled, old fashioned
    335240: "bread",            # bread, whole-wheat, commercially prepared
    2758998: "pasta",           # pasta, dry, enriched, spaghetti
    2706480: "noodles",         # beef and noodles, no sauce
    2707684: "bagel",           # bagel
    2708216: "popcorn",         # popcorn, nfs
    324860: "peanut_butter",    # peanut butter, smooth style, with salt
}

In [244]:
len(target_whole_foods)

100

In [245]:
# Find nutrition for eight whole foods
target_whole_foods_df = food_nutrient[(food_nutrient["fdc_id"].isin(list(whole_foods_id_map.keys()))) & \
    (food_nutrient["nutrient_id"].isin(list(target_nutrient_dict.keys())))][["fdc_id", "nutrient_id", "amount"]]
target_whole_foods_df

,fdc_id,nutrient_id,amount
16925,324860,1003,22.50
16943,324860,1005,22.30
16950,324860,1004,51.10
16980,324860,1008,597.00
16991,324860,2048,597.00
...,...,...,...
461005,2709853,204,0.17
488776,2710281,203,0.30
488795,2710281,205,82.40
488829,2710281,204,0.00


In [246]:
# Pivot the table to how we want it
target_whole_foods_df = target_whole_foods_df.pivot_table("amount", "fdc_id", "nutrient_id")
target_whole_foods_df

nutrient_id,203,204,205,208,1003,1004,1005,1008,2048
fdc_id,,,,,,,,,
324860,NaN,NaN,NaN,NaN,22.500000,51.10000,22.300000,597.0,597.0
325198,NaN,NaN,NaN,NaN,18.000000,30.60000,5.270000,366.0,366.0
325430,NaN,NaN,NaN,NaN,0.910000,0.27000,10.100000,42.0,42.0
327046,NaN,NaN,NaN,NaN,1.060000,0.44000,14.000000,58.0,58.0
327198,NaN,NaN,NaN,NaN,0.820000,0.18000,8.160000,34.0,NaN
...,...,...,...,...,...,...,...,...,...
2747665,NaN,NaN,NaN,NaN,0.656250,0.08313,4.056820,NaN,NaN
2747671,NaN,NaN,NaN,NaN,8.806250,0.55080,0.932150,NaN,NaN
2747673,NaN,NaN,NaN,NaN,24.700000,0.38750,-0.104500,NaN,NaN


In [247]:
target_whole_foods_df.columns

Index([203, 204, 205, 208, 1003, 1004, 1005, 1008, 2048], dtype='int64', name='nutrient_id')

In [248]:
len(whole_foods_id_map)

99

In [249]:
target_whole_foods_df = target_whole_foods_df.reset_index(drop=False).rename_axis(None, axis=1)
target_whole_foods_df

,fdc_id,203,204,205,208,1003,1004,1005,1008,2048
0,324860,NaN,NaN,NaN,NaN,22.500000,51.10000,22.300000,597.0,597.0
1,325198,NaN,NaN,NaN,NaN,18.000000,30.60000,5.270000,366.0,366.0
2,325430,NaN,NaN,NaN,NaN,0.910000,0.27000,10.100000,42.0,42.0
3,327046,NaN,NaN,NaN,NaN,1.060000,0.44000,14.000000,58.0,58.0
4,327198,NaN,NaN,NaN,NaN,0.820000,0.18000,8.160000,34.0,NaN
...,...,...,...,...,...,...,...,...,...,...
90,2747665,NaN,NaN,NaN,NaN,0.656250,0.08313,4.056820,NaN,NaN
91,2747671,NaN,NaN,NaN,NaN,8.806250,0.55080,0.932150,NaN,NaN
92,2747673,NaN,NaN,NaN,NaN,24.700000,0.38750,-0.104500,NaN,NaN
93,2747674,NaN,NaN,NaN,NaN,0.953125,0.11880,7.274975,NaN,NaN


In [250]:
target_nutrient_dict

{1003: 'protein',
 1004: 'fat',
 1005: 'carbohydrate',
 1008: 'energy_kcal',
 2048: 'energy_kcal',
 203: 'protein',
 204: 'fat',
 205: 'carbohydrate',
 208: 'energy_kcal'}

In [251]:
# Rename columns
target_whole_foods_df.rename(columns=target_nutrient_dict, inplace=True)
target_whole_foods_df

,fdc_id,protein,fat,carbohydrate,energy_kcal,protein,fat,carbohydrate,energy_kcal,energy_kcal
0,324860,NaN,NaN,NaN,NaN,22.500000,51.10000,22.300000,597.0,597.0
1,325198,NaN,NaN,NaN,NaN,18.000000,30.60000,5.270000,366.0,366.0
2,325430,NaN,NaN,NaN,NaN,0.910000,0.27000,10.100000,42.0,42.0
3,327046,NaN,NaN,NaN,NaN,1.060000,0.44000,14.000000,58.0,58.0
4,327198,NaN,NaN,NaN,NaN,0.820000,0.18000,8.160000,34.0,NaN
...,...,...,...,...,...,...,...,...,...,...
90,2747665,NaN,NaN,NaN,NaN,0.656250,0.08313,4.056820,NaN,NaN
91,2747671,NaN,NaN,NaN,NaN,8.806250,0.55080,0.932150,NaN,NaN
92,2747673,NaN,NaN,NaN,NaN,24.700000,0.38750,-0.104500,NaN,NaN
93,2747674,NaN,NaN,NaN,NaN,0.953125,0.11880,7.274975,NaN,NaN


In [252]:
whole_foods_id_map

{1750339: 'apple',
 1105314: 'banana',
 2263889: 'blueberries',
 2346409: 'strawberries',
 746771: 'orange',
 2710833: 'mango',
 2346398: 'pineapple',
 2747675: 'watermelon',
 2263890: 'grapes',
 746773: 'pear',
 325430: 'peach',
 327046: 'kiwi',
 2709246: 'papaya',
 2710824: 'avocado',
 2709267: 'pomegranate',
 2346399: 'cherries',
 2263888: 'raspberries',
 2727581: 'blackberries',
 327198: 'cantaloupe',
 2758977: 'grapefruit',
 2710837: 'plum',
 2710815: 'apricot',
 746768: 'figs',
 2709203: 'dates',
 2707501: 'coconut',
 746764: 'carrots',
 1750347: 'mushrooms',
 747447: 'broccoli',
 2685573: 'cauliflower',
 2346407: 'cabbage',
 1750352: 'spinach',
 2346388: 'lettuce',
 2346406: 'cucumber',
 2685578: 'tomato',
 2258588: 'bell_pepper',
 1104962: 'onion',
 1104647: 'garlic',
 2346401: 'potato',
 2346404: 'sweet_potato',
 2727578: 'pumpkin',
 2685568: 'zucchini',
 2685577: 'eggplant',
 2710823: 'asparagus',
 2346405: 'celery',
 2709853: 'green_beans',
 2644291: 'peas',
 2710826: 'corn'

In [253]:
# Filter to target foods and nutrients, then map nutrient names
target_whole_foods_df = food_nutrient[
    (food_nutrient["fdc_id"].isin(list(whole_foods_id_map.keys()))) & 
    (food_nutrient["nutrient_id"].isin(list(target_nutrient_dict.keys())))
][["fdc_id", "nutrient_id", "amount"]].copy()

# Map nutrient IDs to names BEFORE pivoting
target_whole_foods_df["nutrient_name"] = target_whole_foods_df["nutrient_id"].map(target_nutrient_dict)

# Pivot by name — same names get merged into one column, taking first non-null
target_whole_foods_df = target_whole_foods_df.pivot_table(
    "amount", "fdc_id", "nutrient_name", aggfunc="first"
)

# Clean up
target_whole_foods_df = target_whole_foods_df.reset_index(drop=False).rename_axis(None, axis=1)
target_whole_foods_df

,fdc_id,carbohydrate,energy_kcal,fat,protein
0,324860,22.300000,597.0,51.10000,22.500000
1,325198,5.270000,366.0,30.60000,18.000000
2,325430,10.100000,42.0,0.27000,0.910000
3,327046,14.000000,58.0,0.44000,1.060000
4,327198,8.160000,34.0,0.18000,0.820000
...,...,...,...,...,...
90,2747665,4.056820,NaN,0.08313,0.656250
91,2747671,0.932150,NaN,0.55080,8.806250
92,2747673,-0.104500,NaN,0.38750,24.700000
93,2747674,7.274975,NaN,0.11880,0.953125


In [254]:
# Check if a survey food ID exists in food_nutrient
food_nutrient[food_nutrient["fdc_id"] == 2710616]  # beer

,id,fdc_id,nutrient_id,amount,data_points,derivation_id,min,max,median,footnote,min_year_acquired,nutrient_name
510549,34476297,2710616,627,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
510550,34476239,2710616,203,0.46,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
510551,34476267,2710616,401,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
510552,34476292,2710616,618,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
510553,34476280,2710616,578,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
510609,34476261,2710616,322,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
510610,34476287,2710616,611,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
510611,34476249,2710616,301,4.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
510612,34476299,2710616,629,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [255]:
# Show which foods in the map have NO nutrient data
all_ids = set(whole_foods_id_map.keys())
found_ids = set(target_whole_foods_df["fdc_id"].unique())
missing_ids = all_ids - found_ids
for mid in missing_ids:
    print(f"❌ {whole_foods_id_map[mid]:20s} (ID: {mid})")

❌ grapefruit           (ID: 2758977)
❌ ham                  (ID: 2759002)
❌ chicken_breast       (ID: 2759004)
❌ pasta                (ID: 2758998)


In [256]:
# Find which nutrient IDs exist for your missing survey foods
survey_ids_in_map = [k for k in whole_foods_id_map.keys() if str(k).startswith("27")]
survey_nutrients = food_nutrient[food_nutrient["fdc_id"].isin(survey_ids_in_map)]
print("Nutrient IDs available in survey foods:")
print(survey_nutrients["nutrient_id"].unique())

Nutrient IDs available in survey foods:
[1063 2048 2047 1005 1003 1162 1079 1101 1014 1092 1004 1098 1095 1093
 1012 1090 1013 1010 1007 1002 1089 1011 1091 1176 1051 1087 1009 1175
 1165 1167 1177 1166 1253 1178 1103 2020 2018 2016 1329 2014 1273 1312
 2004 1305 1263 1303 1271 1333 1259 1335 1293 1267 1257 1311 1405 2024
 1262 1334 1277 1414 1260 1406 2006 1274 2005 1266 1270 1281 1321 1299
 1301 2013 2025 1265 2019 1276 1275 1411 2021 2026 1258 2008 1300 2003
 1264 1292 2015 2023 2007 2017 1261 1304 1180 1268 1323 1269  630  204
  312  338  301  263  208  406  618  328  203  631  432  645  621  307
  269  626  221  205  627  617  405  430  619  309  303  322  319  573
  417  629  323  334  601  317  337  609  262  613  418  255  612  578
  431  610  608  611  646  607  404  304  614  606  306  620  415  435
  421  321  401  291  320  628  305]


In [257]:
# Add food names
target_whole_foods_df["food_name"] = target_whole_foods_df["fdc_id"].map(whole_foods_id_map)
target_whole_foods_df

,fdc_id,carbohydrate,energy_kcal,fat,protein,food_name
0,324860,22.300000,597.0,51.10000,22.500000,peanut_butter
1,325198,5.270000,366.0,30.60000,18.000000,cheddar_cheese
2,325430,10.100000,42.0,0.27000,0.910000,peach
3,327046,14.000000,58.0,0.44000,1.060000,kiwi
4,327198,8.160000,34.0,0.18000,0.820000,cantaloupe
...,...,...,...,...,...,...
90,2747665,4.056820,NaN,0.08313,0.656250,radish
91,2747671,0.932150,NaN,0.55080,8.806250,squid
92,2747673,-0.104500,NaN,0.38750,24.700000,tuna
93,2747674,7.274975,NaN,0.11880,0.953125,turnip


All amounts are per 100g.

In [260]:
# Show all foods with any NaN
nan_foods = target_whole_foods_df[target_whole_foods_df.isna().any(axis=1)]
nan_foods[["food_name", "protein", "fat", "carbohydrate", "energy_kcal"]]

,food_name,protein,fat,carbohydrate,energy_kcal
16,butter,NaN,82.2,NaN,739.8000
87,pumpkin,0.854375,NaN,NaN,3.4175
88,blackberries,1.526250,NaN,NaN,6.1050
94,watermelon,0.871250,NaN,NaN,3.4850


In [259]:
# Calculate missing energy_kcal using: 4*protein + 4*carbs + 9*fat
mask = target_whole_foods_df["energy_kcal"].isna()
target_whole_foods_df.loc[mask, "energy_kcal"] = (
    4 * target_whole_foods_df.loc[mask, "protein"].fillna(0) +
    4 * target_whole_foods_df.loc[mask, "carbohydrate"].fillna(0) +
    9 * target_whole_foods_df.loc[mask, "fat"].fillna(0)
)

# Show which foods were fixed
fixed = target_whole_foods_df.loc[mask, ["food_name", "energy_kcal"]]
if len(fixed) > 0:
    print(f"✅ Fixed {len(fixed)} foods with calculated energy_kcal:")
    for _, row in fixed.iterrows():
        print(f"   {row['food_name']:20s} → {row['energy_kcal']:.1f} kcal")
else:
    print("✅ No missing energy values to fix!")

✅ Fixed 9 foods with calculated energy_kcal:
   butter               → 739.8 kcal
   pumpkin              → 3.4 kcal
   blackberries         → 6.1 kcal
   lobster              → 58.8 kcal
   radish               → 19.6 kcal
   squid                → 43.9 kcal
   tuna                 → 101.9 kcal
   turnip               → 34.0 kcal
   watermelon           → 3.5 kcal


## Export first 10 target food nutrition information

In [261]:
target_whole_foods_df.to_csv("target_hundred_whole_food_nutrition_info.csv", index=False)

In [180]:
ten_whole_foods

['chicken_wings',
 'apple',
 'banana',
 'beef',
 'carrots',
 'egg',
 'strawberries',
 'blueberries',
 'mushrooms',
 'honey']

In [181]:
foundation_food.head(-10)

,fdc_id,data_type,description,food_category_id,publication_date
651,321358,foundation_food,"hummus, commercial",16.0,2019-04-01
652,321359,foundation_food,"milk, reduced fat, fluid, 2% milkfat, with add...",1.0,2019-04-01
653,321360,foundation_food,"tomatoes, grape, raw",11.0,2019-04-01
798,321505,foundation_food,"salt, table, iodized",2.0,2019-04-01
904,321611,foundation_food,"beans, snap, green, canned, regular pack, drai...",11.0,2019-04-01
...,...,...,...,...,...
93407,2710800,survey_fndds_food,"cabbage, cooked, as ingredient",9999.0,2022-10-28
93408,2710801,survey_fndds_food,"cauliflower, cooked, as ingredient",9999.0,2022-10-28
93409,2710802,survey_fndds_food,"eggplant, cooked, as ingredient",9999.0,2022-10-28
93410,2710803,survey_fndds_food,"green beans, cooked, as ingredient",9999.0,2022-10-28


# Mine

In [52]:
foundation_food["data_type"].unique()

<ArrowStringArray>
['foundation_food', 'survey_fndds_food']
Length: 2, dtype: str

In [53]:
foundation_food["description"].unique()

<ArrowStringArray>
[                                                        'hummus, commercial',
   'milk, reduced fat, fluid, 2% milkfat, with added vitamin a and vitamin d',
                                                       'tomatoes, grape, raw',
                                                       'salt, table, iodized',
                   'beans, snap, green, canned, regular pack, drained solids',
                                                              'broccoli, raw',
        'milk, lowfat, fluid, 1% milkfat, with added vitamin a and vitamin d',
 'milk, nonfat, fluid, with added vitamin a and vitamin d (fat free or skim)',
                           'milk, whole, 3.25% milkfat, with added vitamin d',
                                                'frankfurter, beef, unheated',
 ...
                                         'cauliflower, cooked, as ingredient',
                                            'eggplant, cooked, as ingredient',
                            

In [54]:
foundation_foods[50:100]

10162    turkey, ground, 93% lean, 7% fat, pan-broiled ...
11190    chicken, broilers or fryers, drumstick, meat o...
11253    chicken, broiler or fryers, breast, skinless, ...
11575     sauce, pasta, spaghetti/marinara, ready-to-serve
11690    ham, sliced, pre-packaged, deli meat (96%fat f...
11890    pears, raw, bartlett (includes foods for usda'...
12084     olives, green, manzanilla, stuffed with pimiento
12157    sausage, pork, chorizo, link or ground, cooked...
12301                 cookies, oatmeal, soft, with raisins
12574                   tomatoes, canned, red, ripe, diced
12667                                   fish, haddock, raw
12769                                   fish, pollock, raw
13487    fish, tuna, light, canned in water, drained so...
13540                                   sugars, granulated
13755             restaurant, chinese, sweet and sour pork
13829        restaurant, chinese, fried rice, without meat
13921                     restaurant, latino, tamale, po

In [55]:
for food in foundation_foods:
    if "chicken" in food.lower():
        print(food)

chicken, broilers or fryers, drumstick, meat only, cooked, braised
chicken, broiler or fryers, breast, skinless, boneless, meat only, cooked, braised
mock chicken legs, cooked
chicken, ns as to part and cooking method, ns as to skin eaten
chicken, ns as to part and cooking method, skin eaten
chicken, ns as to part and cooking method, skin not eaten
chicken, ns as to part, baked, broiled, or roasted, ns as to skin eaten
chicken, ns as to part, baked, broiled, or roasted, skin eaten
chicken, ns as to part, baked, broiled, or roasted, skin not eaten
chicken, ns as to part, rotisserie, ns as to skin eaten
chicken, ns as to part, rotisserie, skin eaten
chicken, ns as to part, rotisserie, skin not eaten
chicken, ns as to part, stewed, ns as to skin eaten
chicken, ns as to part, stewed, skin eaten
chicken, ns as to part, stewed, skin not eaten
chicken, ns as to part, grilled without sauce, ns as to skin eaten
chicken, ns as to part, grilled without sauce, skin eaten
chicken, ns as to part, gr

In [56]:
foundation_food[foundation_food["description"].str.contains("chicken", case=False)]

,fdc_id,data_type,description,food_category_id,publication_date
11190,331897,foundation_food,"chicken, broilers or fryers, drumstick, meat o...",5.0,2019-04-01
11253,331960,foundation_food,"chicken, broiler or fryers, breast, skinless, ...",5.0,2019-04-01
28471,1098388,survey_fndds_food,"mock chicken legs, cooked",NaN,2020-10-30
28501,1098418,survey_fndds_food,"chicken, ns as to part and cooking method, ns ...",NaN,2020-10-30
28502,1098419,survey_fndds_food,"chicken, ns as to part and cooking method, ski...",NaN,2020-10-30
...,...,...,...,...,...
33874,1103791,survey_fndds_food,"vegetable and chicken, baby food, ns as to str...",NaN,2020-10-30
33875,1103792,survey_fndds_food,"vegetable and chicken, baby food, strained",NaN,2020-10-30
33876,1103793,survey_fndds_food,"vegetable and chicken, baby food, junior",NaN,2020-10-30
33883,1103800,survey_fndds_food,"potato chicken pie, puerto rican style",NaN,2020-10-30


In [57]:
nutrient[nutrient["name"].str.contains("protein", case=False) | \
         nutrient["name"].str.contains("carbohydrate", case=False) | \
         nutrient["name"].str.contains("fat", case=False)]["name"].unique()

<ArrowStringArray>
[                                                             'Protein',
                                                    'Total lipid (fat)',
                                          'Carbohydrate, by difference',
                                                      'Solids, non-fat',
                                           'Carbohydrate, by summation',
                                                     'Adjusted Protein',
                                                  'Carbohydrate, other',
                                                     'Total fat (NLEA)',
                                             'Fatty acids, total trans',
                                         'Fatty acids, total saturated',
 'Fatty acids, other than 607-615, 617-621, 624-632, 652-654, 686-689)',
                                   'Fatty acids, total monounsaturated',
                                   'Fatty acids, total polyunsaturated',
                                

Getting IDs of Carbohydrate, Fat and Protein

* Carbohydrate, by difference = total carbohydrate

In [58]:
target_nutrients = nutrient[nutrient["name"].isin(["Protein", "Carbohydrate, by difference", "Total lipid (fat)"])]
target_nutrients

,id,name,unit_name,nutrient_nbr,rank
2,1003,Protein,G,203.0,600.0
3,1004,Total lipid (fat),G,204.0,800.0
4,1005,"Carbohydrate, by difference",G,205.0,1110.0


In [59]:
food_nutrient[(food_nutrient["fdc_id"] ==  chicken_wing_id) & (food_nutrient["nutrient_id"].isin(list(target_nutrient_dict.keys())))]

,id,fdc_id,nutrient_id,amount,data_points,derivation_id,min,max,median,footnote,min_year_acqured,sf.footnote,min_year_acquired,nutrient_name
41686,2259098,331897,1004,5.95,6.0,1.0,5.54,6.33,5.93,NaN,2010.0,NaN,NaN,total lipid (fat)
41718,2259079,331897,1003,23.90,NaN,49.0,23.00,24.60,24.10,NaN,NaN,NaN,NaN,protein
41729,2259099,331897,1005,0.00,NaN,49.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"carbohydrate, by difference"


Getting Target Nutrient(protein, total lipid(fat), carbohydrate, by difference) of `chicken_wing_id`

In [74]:
food_nutrient[(food_nutrient["fdc_id"] == chicken_wing_id) & food_nutrient["nutrient_id"].isin(list(target_nutrient_dict.keys()))]

,id,fdc_id,nutrient_id,amount,data_points,derivation_id,min,max,median,footnote,min_year_acqured,sf.footnote,min_year_acquired,nutrient_name
41686,2259098,331897,1004,5.95,6.0,1.0,5.54,6.33,5.93,NaN,2010.0,NaN,NaN,total lipid (fat)
41718,2259079,331897,1003,23.90,NaN,49.0,23.00,24.60,24.10,NaN,NaN,NaN,NaN,protein
41729,2259099,331897,1005,0.00,NaN,49.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"carbohydrate, by difference"


In [95]:
food[(food["description"].str.contains("blueberry", case=False))]["description"]

31153    pie, berry, not blackberry, blueberry, boysenb...
31154    pie, berry, not blackberry, blueberry, boysenb...
31155    pie, berry, not blackberry, blueberry, boysenb...
31156                            pie, blueberry, two crust
31157              pie, blueberry, individual size or tart
31239                                     crisp, blueberry
31848           cereal (malt-o-meal blueberry muffin tops)
31888               cereal (kellogg's special k blueberry)
32788                                blueberry pie filling
32833                                      blueberry juice
32939        blueberry yogurt dessert, baby food, strained
34033                                      blueberry syrup
Name: description, dtype: str

In [100]:
# food_nutrient[(food_nutrient["fdc_id"] == chicken_curry_id) & food_nutrient["nutrient_id"].isin(list(target_nutrient_dict.keys()))]
chicken_curry_id = int(foundation_food.loc[foundation_food["description"].str.contains(pattern, case=False)].iloc[0]["fdc_id"])
chicken_curry_id

323121

**Code Snip!!!**

Setting Food Id \
`chicken_wing_id = int(foundation_food.loc[foundation_food["description"].str.contains("Chicken", case=False)].iloc[0]["fdc_id"])`

Getting Nutrition Data from ID \
`food_nutrient[(food_nutrient["fdc_id"] == chicken_wing_id) & food_nutrient["nutrient_id"].isin(list(target_nutrient_dict.keys()))]`

In [82]:
chicken_curry_id = int(foundation_food.loc[foundation_food["description"].str.contains("chicken curry", case=False)].iloc[0]["fdc_id"])
food_nutrient[(food_nutrient["fdc_id"] == chicken_curry_id) & food_nutrient["nutrient_id"].isin(list(target_nutrient_dict.keys()))]

,id,fdc_id,nutrient_id,amount,data_points,derivation_id,min,max,median,footnote,min_year_acqured,sf.footnote,min_year_acquired,nutrient_name
218537,12988060,1099246,1005,6.47,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"carbohydrate, by difference"
218540,12988058,1099246,1003,6.47,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,protein
218555,12988059,1099246,1004,6.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,total lipid (fat)


In [172]:
# Check if 2048 nutrient entries exist for our target foods
fdc_ids_with_2048 = food_nutrient[
    (food_nutrient["fdc_id"].isin(list(whole_foods_id_map.keys()))) &
    (food_nutrient["nutrient_id"] == 2048)
]
print(f"Foods with Energy (Atwater Specific Factors) recorded: {len(fdc_ids_with_2048)}")
fdc_ids_with_2048[["fdc_id", "amount"]].merge(
    pd.DataFrame(whole_foods_id_map.items(), columns=["fdc_id", "food_name"]),
    on="fdc_id",
    how="right"
)

Foods with Energy (Atwater Specific Factors) recorded: 2


,fdc_id,amount,food_name
0,1750339,55.622745,apple
1,1105314,NaN,banana
2,1102702,NaN,blueberries
3,746763,NaN,beef
4,746764,NaN,carrots
5,331897,NaN,chicken_wings
6,329490,NaN,egg
7,1103956,NaN,honey
8,1750347,24.873258,mushrooms
9,747448,NaN,strawberries
